# Train Confidence Model
This notebook trains an EfficientNet-B0 classification model using the dataset provided. It is meant to detect confidence (`confident`, `nervous`, `neutral`) from candidate facial crops.

In [ ]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms, models
from PIL import Image
from pathlib import Path
import requests
import zipfile

## 1. Download Dataset

In [ ]:
# NOTE: Insert the dataset link the user provided below. 
DATASET_URL = "YOUR_DATASET_URL_HERE"
DATA_DIR = Path("../../data/confidence_dataset")

def download_and_extract(url, out_dir):
    if not out_dir.exists():
        out_dir.mkdir(parents=True, exist_ok=True)
        print(f"Downloading dataset from {url}...")
        # Example download logic:
        # response = requests.get(url, stream=True)
        # zip_path = out_dir / "dataset.zip"
        # with open(zip_path, 'wb') as f:
        #     for chunk in response.iter_content(chunk_size=8192):
        #         f.write(chunk)
        # print("Extracting...")
        # with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        #     zip_ref.extractall(out_dir)
        # zip_path.unlink()
        print("Done.")
    else:
        print("Dataset already exists.")

if DATASET_URL and DATASET_URL != "YOUR_DATASET_URL_HERE":
    download_and_extract(DATASET_URL, DATA_DIR)

## 2. Prepare DataLoaders

In [ ]:
class ConfidenceDataset(Dataset):
    def __init__(self, data_dir, split="train", transform=None):
        self.data_dir = Path(data_dir) / split
        self.transform = transform
        self.classes = ['confident', 'nervous', 'neutral']
        self.images = []
        self.labels = []
        
        # Load images
        if self.data_dir.exists():
            for i, cls in enumerate(self.classes):
                cls_dir = self.data_dir / cls
                if cls_dir.exists():
                    for img_path in cls_dir.glob("*.jpg"):
                        self.images.append(img_path)
                        self.labels.append(i)

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img_path = self.images[idx]
        label = self.labels[idx]
        img = Image.open(img_path).convert("RGB")
        if self.transform:
            img = self.transform(img)
        return img, label

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# train_ds = ConfidenceDataset(DATA_DIR, "train", train_transform)
# val_ds = ConfidenceDataset(DATA_DIR, "val", val_transform)
# train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
# val_loader = DataLoader(val_ds, batch_size=32, shuffle=False)

## 3. Model Definition & Training Loop

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu"))
print(f"Using device: {device}")

model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)
model.classifier[1] = nn.Linear(model.classifier[1].in_features, 3)
model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)

def train_model(epochs=10):
    print("Starting training...")
    # Uncomment when dataset is ready 
    '''
    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
            
        print(f"Epoch {epoch+1}/{epochs} - Loss: {running_loss/len(train_loader):.4f}")
    '''
    print("Training Complete!")

train_model(epochs=10)

## 4. Save Weights

In [ ]:
out_dir = Path("weights")
out_dir.mkdir(parents=True, exist_ok=True)
torch.save(model.state_dict(), out_dir / "confidence_model.pth")
print("Model saved to ai_models/confidence/weights/confidence_model.pth")